# AI Battery Safety Guardian — M11 LoRA Explainer (Qwen-2.5-0.5B PEFT)

This notebook fine-tunes a PEFT LoRA adapter on `Qwen/Qwen2.5-0.5B-Instruct` for structured battery safety assessment explanation on an **NVIDIA T4 GPU** in Google Colab.

### Pipeline Structure:
- **Cell 1**: GPU Check & Validation
- **Cell 2**: Upload `colab_pack.zip` & Install Dependencies (with torchao conflict fix)
- **Cell 3**: Fast Smoke Test (CPU/GPU pipeline sanity check)
- **Cell 4**: Full Training Run (fp16, LoRA r=16, alpha=32, checkpointing every 50 steps, resume support)
- **Cell 5**: Base vs Adapter Comparison on Validation Set, Bundle `results_lora.zip` & Auto-Download

In [ ]:
# Cell 1: Check GPU environment
import torch

if not torch.cuda.is_available():
    raise SystemError("❌ GPU not detected! In Google Colab, go to 'Runtime' -> 'Change runtime type' and select 'T4 GPU'.")

gpu_name = torch.cuda.get_device_name(0)
print(f"✅ Active GPU: {gpu_name}")
print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")
print(f"Allocated VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Cell 2: Upload colab_pack.zip and install dependencies
import os
import zipfile
from google.colab import files

if not os.path.exists('colab_pack.zip'):
    print("Please select and upload 'colab_pack.zip' from your local project:")
    uploaded = files.upload()

print("Extracting colab_pack.zip...")
with zipfile.ZipFile('colab_pack.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

print("Installing dependencies from requirements-colab.txt...")
# Uninstall incompatible Colab default torchao if present
!pip uninstall -y -q torchao
!pip install -q -r requirements-colab.txt
print("✅ Setup complete!")

In [ ]:
# Cell 3: Fast smoke test (5 steps)
print("Running smoke test to verify model loading and forward/backward passes...")
!python scripts/train_lora.py \
    --data-dir data/lora \
    --smoke \
    --output-dir artifacts/models/lora_explainer_smoke \
    --metrics-out artifacts/metrics/lora_smoke.json

print("✅ Smoke test succeeded without errors!")

In [ ]:
# Cell 4: Full training run on T4 GPU (supports --resume if disconnected)
print("Starting LoRA training (2 epochs, checkpointing every 50 steps)...")
!python scripts/train_lora.py \
    --data-dir data/lora \
    --output-dir artifacts/models/lora_explainer \
    --metrics-out artifacts/metrics/lora.json \
    --train-log-out artifacts/metrics/train_log.json \
    --samples-out artifacts/metrics/lora_samples.jsonl \
    --epochs 2 \
    --resume

import json
with open('artifacts/metrics/lora.json') as f:
    metrics = json.load(f)

print("\n--- LoRA EVALUATION RESULTS ---")
print(f"Base Model:               {metrics['base_model']}")
print(f"Training Duration:        {metrics['train_duration_s']}s")
print(f"Adapter ROUGE-L:          {metrics['evaluation']['adapter_model']['mean_rouge_l']:.4f}")
print(f"Base Model ROUGE-L:       {metrics['evaluation']['base_model']['mean_rouge_l']:.4f}")
print(f"JSON Validity Rate:       {metrics['evaluation']['adapter_model']['json_validity_rate']:.1%}")
print(f"Forbidden Violations:     {metrics['evaluation']['adapter_model']['forbidden_violation_rate']:.1%}")

In [ ]:
# Cell 5: Create results bundle and auto-download
import shutil
from google.colab import files

bundle_name = 'results_lora.zip'
export_dir = 'export_lora'
os.makedirs(f'{export_dir}/artifacts/models/lora_explainer', exist_ok=True)
os.makedirs(f'{export_dir}/artifacts/metrics', exist_ok=True)

# Copy adapter folder ONLY (not the full base model)
shutil.copytree('artifacts/models/lora_explainer', f'{export_dir}/artifacts/models/lora_explainer', dirs_exist_ok=True)
# Copy metrics
shutil.copy2('artifacts/metrics/lora.json', f'{export_dir}/artifacts/metrics/lora.json')
# Copy loss curve train_log.json
if os.path.exists('artifacts/metrics/train_log.json'):
    shutil.copy2('artifacts/metrics/train_log.json', f'{export_dir}/artifacts/metrics/train_log.json')
# Copy val set sample comparisons
if os.path.exists('artifacts/metrics/lora_samples.jsonl'):
    shutil.copy2('artifacts/metrics/lora_samples.jsonl', f'{export_dir}/artifacts/metrics/lora_samples.jsonl')

# Create zip
shutil.make_archive('results_lora', 'zip', export_dir)
print(f"✅ Bundle '{bundle_name}' created successfully ({os.path.getsize(bundle_name) / 1e6:.2f} MB).")
print("Downloading results bundle to your local machine...")
files.download(bundle_name)